# eval DB viewer

Read-only look at the two shared SQLite DBs.

- **records (ops DB)** `easyads_ops.db` — jobs / node_executions / llm_calls / schema_validations / job_cost_summary / dirty_field_events
- **eval DB** `easyads_eval.db` — eval_runs / score_items / gate_results / domain_scores / judge_status

Run the **Setup** cell first, then any cell below. Re-run a cell to refresh (safe to run while `make eval-sample` is generating).

**Validity rule:** a job is real only if every `llm_calls.fallback_used = 0` AND `t2i_engine = gpt_image_2` AND `imgs >= 1`. Any `fallback_used = 1` -> discard that job.

## Setup

In [ ]:
import os, sqlite3
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 80)

# /home/records on host, /app/records inside the orchestrator container. Override with RECORDS_DIR.
RECORDS = os.environ.get('RECORDS_DIR') or next(
    (p for p in ('/home/records', '/app/records') if os.path.isdir(p)), '/home/records')
OPS_DB  = os.path.join(RECORDS, 'easyads_ops.db')
EVAL_DB = os.path.join(RECORDS, 'easyads_eval.db')

def q(db, sql, params=()):
    """Run a read-only query, return a DataFrame."""
    with sqlite3.connect(db) as c:
        return pd.read_sql_query(sql, c, params=params)

print('RECORDS :', RECORDS)
print('OPS_DB  :', OPS_DB,  '(exists)' if os.path.isfile(OPS_DB)  else '(MISSING)')
print('EVAL_DB :', EVAL_DB, '(exists)' if os.path.isfile(EVAL_DB) else '(MISSING)')

# records (ops DB)

### jobs — overview (newest first)

In [ ]:
q(OPS_DB, '''
  SELECT job_id, user_plan AS plan, render_profile AS render, status,
         entry_mode, revision, created_at
  FROM jobs ORDER BY created_at DESC LIMIT 30
''')

### llm_calls — fallback summary per job (fb_total must be 0 for a valid run)

In [ ]:
q(OPS_DB, '''
  SELECT job_id,
         COUNT(*)            AS llm_calls,
         SUM(fallback_used)  AS fb_total,
         SUM(success)        AS ok_total,
         GROUP_CONCAT(DISTINCT error_code) AS errors
  FROM llm_calls GROUP BY job_id ORDER BY job_id DESC LIMIT 30
''')

### llm_calls — per-node detail (model / fallback / error / cost)

In [ ]:
q(OPS_DB, '''
  SELECT job_id, node_name, model_class, model_name,
         success AS ok, fallback_used AS fb,
         COALESCE(error_code,'') AS error_code,
         total_tokens AS tok, cost_usd, cost_source
  FROM llm_calls ORDER BY id DESC LIMIT 60
''')

### node_executions — status & latency (t2i_generation dominates wall time)

In [ ]:
# set a job_id to focus, or leave '' for the latest job
JOB_ID = ''
if not JOB_ID:
    JOB_ID = q(OPS_DB, 'SELECT job_id FROM jobs ORDER BY created_at DESC LIMIT 1').iloc[0,0]
print('JOB_ID =', JOB_ID)
q(OPS_DB, '''
  SELECT node_name, status, latency_ms AS ms,
         SUBSTR(COALESCE(error_message,''),1,50) AS err
  FROM node_executions WHERE job_id=? ORDER BY id
''', (JOB_ID,))

### job_cost_summary — tokens, USD, T2I engine & image cost

In [ ]:
q(OPS_DB, '''
  SELECT job_id, total_api_calls AS api, fallback_calls AS fb,
         total_tokens AS tok, total_cost_usd AS usd,
         t2i_engine, t2i_image_count AS imgs, t2i_cost_usd, t2i_cost_source,
         cost_estimated AS est
  FROM job_cost_summary ORDER BY rowid DESC LIMIT 30
''')

### images — generated file paths (result node)

In [ ]:
q(OPS_DB, '''
  SELECT ne.job_id,
         json_extract(nso.output_snapshot,'$.result_payload.output_path')           AS final_ad,
         json_extract(nso.output_snapshot,'$.result_payload.background_image_path')  AS background
  FROM node_state_outputs nso
  JOIN node_executions ne ON ne.id = nso.node_exec_id
  WHERE ne.node_name='result'
  GROUP BY ne.job_id ORDER BY ne.id DESC LIMIT 30
''')

### schema_validations — failures only (empty = all passed)

In [ ]:
q(OPS_DB, '''
  SELECT job_id, node_name, schema_name, field_name,
         SUBSTR(COALESCE(error_detail,''),1,80) AS err
  FROM schema_validations WHERE passed=0 ORDER BY id DESC LIMIT 50
''')

### preview a generated image (optional)

In [ ]:
from IPython.display import Image, display
row = q(OPS_DB, '''
  SELECT json_extract(nso.output_snapshot,'$.result_payload.output_path') AS final_ad
  FROM node_state_outputs nso JOIN node_executions ne ON ne.id=nso.node_exec_id
  WHERE ne.node_name='result' ORDER BY ne.id DESC LIMIT 1
''')
path = row.iloc[0,0] if len(row) else None
print('path:', path)
if path and os.path.isfile(path):
    display(Image(filename=path))
else:
    print('file not reachable from here (may be a container path); open it directly.')

# eval DB

### eval_runs — overall score & verdict

In [ ]:
q(EVAL_DB, '''
  SELECT job_id, eval_id, ROUND(overall_score,3) AS score, verdict,
         evaluator_id, evaluated_at
  FROM eval_runs ORDER BY evaluated_at DESC LIMIT 30
''')

### score_items — per-item scores by evaluator (llm / vlm / human)

In [ ]:
q(EVAL_DB, '''
  SELECT er.job_id, si.evaluator_type, si.item_id, si.score,
         SUBSTR(COALESCE(si.notes,''),1,60) AS notes
  FROM score_items si JOIN eval_runs er ON er.eval_id = si.eval_id
  ORDER BY er.evaluated_at DESC, si.evaluator_type, si.item_id LIMIT 80
''')

### gate_results — auto gates pass/fail

In [ ]:
q(EVAL_DB, '''
  SELECT er.job_id, gr.gate_id, gr.passed,
         SUBSTR(COALESCE(gr.failure_reason,''),1,70) AS failure_reason,
         gr.auto_evaluated
  FROM gate_results gr JOIN eval_runs er ON er.eval_id = gr.eval_id
  ORDER BY er.evaluated_at DESC, gr.gate_id LIMIT 60
''')

### domain_scores — aggregated per domain

In [ ]:
q(EVAL_DB, '''
  SELECT er.job_id, ds.domain, ROUND(ds.avg_score,3) AS avg_score,
         ds.weight, ROUND(ds.weighted_contribution,3) AS weighted, ds.item_count
  FROM domain_scores ds JOIN eval_runs er ON er.eval_id = ds.eval_id
  ORDER BY er.evaluated_at DESC, ds.domain LIMIT 60
''')

### judge_status — idempotent judge queue (attempts / errors)

In [ ]:
q(EVAL_DB, '''
  SELECT eval_id, evaluator_type, status, attempts,
         SUBSTR(COALESCE(last_error,''),1,70) AS last_error
  FROM judge_status ORDER BY rowid DESC LIMIT 40
''')